# 94-867/19-867/12-768 
# From Data to Action: Designing Human-Centered, Impact-Driven Decision Systems

Student Name: `<FILL IN>`

Andrew ID: `<FILL IN>`

## Read This Carefully Before Start

Make sure write your name and andrew id above.

Make sure you FILL IN any place that instructs `YOUR CODE HERE`, `<FILL IN>`, or `YOUR ANSWER HERE`.

Questions entitled with a ✰ symbol means you have to include answers in your written submission. For your convenience, numbers of those questions are consistent with the handout PDF.

Before you turn these assignments in, make sure everything runs as expected. First, **Restart** the kernel and then **Run All** cells.

---

# Assignment 1-3: Duolingo Template Selection via Multi-Armed Bandits

In [ ]:
# Import necessary libraries
# You should not need and are not allowed to import any other libraries

import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Define Problem Setup

In this problem, we adapt the business context of Duolingo notification templates introduced in the September 3 lecture. To keep the problem tractable, we simplify it to use conventional bandit algorithms such as **ε-Greedy** and **Upper Confidence Bound** (UCB) for dynamically choosing notification templates. 

Instead of optimizing template selection at the individual user level, we now focus on group-level optimization. Specifically, a group of 100 users is tested on **20 templates (arms)**, where each user is pushed 30 notifications (roughly a one-month volume), resulting in **3000 total steps**, which ensures a sufficient number of steps for algorithm evaluation. For fairness, all algorithms start from the same initialization: each of the 20 templates is pulled 5 times, totaling **100 initial steps**. After this initialization, the remaining 2900 steps allow the algorithms to adapt and diverge. 

We assume that all users share identical interests and therefore exhibit the same response patterns. In addition, we assume that any notification generated from a given template yields the same expected reward, regardless of when or how often it is delivered. As a result, although each timestep corresponds to sending a notification to a single user within the group, the performance of each template is invariant across both users and time.

*(While out of scope of this problem, for your reference, defining such a user group can be achieved by many unsupervised clustering models.)*

In [ ]:
# Let's define this setup
# ! DO NOT MODIFY THE FOLLOWING VARIABLES
action_space = 20
T = 3000
n_initial_per_arm = 5
n_steps = T - n_initial_per_arm * action_space

### Load Synthesized True Rewards

Instead of defining a reward function, we provide a synthesized **true reward** matrix with a shape of 5,000 (steps) × 20 (actions), where each row represents the responses at one step and each column corresponds to a candidate action (template). In this matrix, an entry value of 1 indicates that a user clicks a notification generated from the corresponding template, while 0 indicates no click. 

For evaluation, we use the click-through rate (CTR), defined as the total number of clicks divided by the total number of pushes. 

It is important to note that this reward matrix is artificially constructed to show responses for all possible actions. In real-world practice, such complete information is NEVER AVAILABLE. In an online MAB setting, **you select one action and only observe the reward of that chosen action, while the outcomes of all other actions remain unknown.**

In [ ]:
# Load the true rewards
true_rewards = np.load('Data/synthesized_template_rewards.npy')
true_rewards

For example, the first row of the synthesized reward matrix
\begin{bmatrix} 0 & 1 & 0 & 0 & 0 & 0 & 0 & 1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 \end{bmatrix}
means that, at step 1, notifications generated from those templates 1, 7, 9, or 19 would be clicked.

Note that we count templates starting from 0, consistent with array indexing.

***How An Action Works***

The action and reward at each timestep are represented as 1D arrays of length 20, respectively.

**Action**: An entry takes the value 1, indicating the selected template, while all other entries remain 0. For example: if you decide to pull template 1 at the first step, the corresponding action array is:
\begin{bmatrix} 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \end{bmatrix}

**Reward**: It works the same as the true rewards, where each entry indicates whether the corresponding template would produce a click (1) or not (0). However, the learner only observes the reward of the chosen action. To encode this, the reward array you record has a single reward entry: the reward value of the template actually selected. All other entries remain 0, since those outcomes are unobserved. The reward array of the example above should be:
\begin{bmatrix} 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \end{bmatrix}

Finally, store the action and reward array in a structured way. To track the interaction history over multiple steps, both the actions and rewards are maintained as 2D arrays, with each row corresponds to one timestep. At the end, the shape of actions and rewards should be the same as the true reward matrix.

### Initialize Environment

Now, with all setups above, let's make actions and get rewards for the first 100 initial steps.

In [ ]:
initial_actions = np.repeat(np.eye(action_space), n_initial_per_arm, axis=0)
initial_rewards = true_rewards[:initial_actions.shape[0]] * initial_actions

### ✰ 3.2(a) Which template to select for the next step under a **full exploitation** policy?

- Write your code for this question in the cell below
- Find the index of the action 
- Choose your response for MCQ Question 3.2(a) in the written submission

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()  # You can comment this line after implementation

## Develop MAB Algorithms

### ✰ 3.2(b) Complete `update()` function in `MAB` class and report it in the written submission

This `MAB` class serves as a base (parent) class from which all specific algorithms (e.g., random, ε-Greedy, UCB) inherit. It provides a common framework for action selection, reward updating, and history tracking. The design follows principles of object-oriented programming (OOP) that you should be familiar with from other coursework.

In [ ]:
class MAB():
    def __init__(self, initial_actions: np.ndarray, initial_rewards: np.ndarray):
        # ! DO NOT MODIFY THE FOLLOWING VARIABLES. You will use them in your implementation.
        """
        Initialize the Multi-Armed Bandit (MAB) environment with the first 100 actions and rewards.
        """
        self.action_space = initial_actions.shape[1]  # 20 templates for consideration
        self.n_init_steps = initial_actions.shape[0]  # 100 steps already taken
        self.actions = initial_actions  # History of all actions taken, should be updated over time
        self.rewards = initial_rewards  # History of all rewards received, should be updated over time

    def plot_avg_reward(self, label=''):
        # ! DO NOT MODIFY THIS FUNCTION
        """
        We provide this function to plot the average reward over time, for your convenience.
        While this function should not be modified, you are free to build your own for visualization as needed.

        Args:
            label (str): Name of the algorithm shown in the plot legend.
        """
        t_s = [t+1 for t in range(len(self.rewards))]  # 1, 2, 3, ..., T
        # ? Think about the shape of average reward for plot
        avg_rewards = [self.rewards[:t+1].sum() / (t+1)
                       for t in range(len(self.rewards))]

        plt.plot(t_s, avg_rewards, label=label)
        plt.xlabel("Timestep")
        plt.ylabel("Average Reward")
        plt.legend()
        plt.grid(True, linestyle='--')

    def step(self) -> int:
        #! DO NOT MODIFY THIS FUNCTION
        """
        Select an action and return its index.
        This function serves as a placeholder and is intended to be overridden by subclasses.
        For placeholder purposes, we simply select the first action for every timestep.

        Returns:
            int: Index of the selected action.
        """
        return 0

    def update(self, true_rewards: np.ndarray, n_steps: int):
        # TODO: COMPLETE THIS FUNCTION
        """
        Given a number of steps, at each timestep:
            take an action using the current policy via self.step();
            receive the corresponding reward from the provided true reward matrix;
            update the stored action and reward history.

        Args:
            true_rewards (np.ndarray): The synthsized true reward matrix.
            n_steps (int): The number of steps to update.
        """
        for t in range(n_steps):
            action_index = self.step()
            # YOUR CODE HERE
            raise NotImplementedError()


# ! DO NOT MODIFY THE FOLLOWING LINES FOR TESTING
toy = MAB(initial_actions, initial_rewards)
toy.update(true_rewards, n_steps)

# A successful implementation of update() function should pass the following test cases
assert toy.actions.shape == (T, action_space), "Incorrect shape for action history"
assert toy.rewards.shape == (T, action_space), "Incorrect shape for reward history"
assert toy.rewards.sum() == 911, "Incorrect total reward received after 3000 steps"

With the successfully implemented toy model, let's plot its average reward over time using the provided `plot_avg_reward()` function.

In [ ]:
toy.plot_avg_reward(label='Toy')

The `FullRandom` class below illustrates how a child class (also referred to as a subclass) extends the base `MAB` class. It inherits all attributes and methods from `MAB` but overrides the `step()` function to implement a random action-selection strategy. Other algorithms (e.g., ε-Greedy, UCB) can be implemented within the same framework by similarly overriding `step()` while reusing the shared functionality of the parent class.

You don't need to do anything in the cell below.

Simply run the cell below to see **how a child class in OOP works** and to see a **sample plot of algorithm comparsion**.

In [ ]:
class FullRandom(MAB):
    def __init__(self, initial_actions: np.ndarray, initial_rewards: np.ndarray):
        """
        # With the same input parameters as MAB, initialize the parent class
        """
        super().__init__(initial_actions, initial_rewards)

    def step(self) -> int:
        """
        Select an action uniformly at random.
        It overrides the step function in the MAB class.

        Returns:
            int: Index of the selected action.
        """
        # Return a random index from 0 to 19, instead of fixed 0 in the parent class
        return random.randint(0, self.action_space - 1)


# Implement the random policy
full_random = FullRandom(initial_actions, initial_rewards)
full_random.update(true_rewards, n_steps)

# Plot toy and random model for comparison
toy.plot_avg_reward(label='Toy')
full_random.plot_avg_reward(label='Random')

### ✰ 3.2(c) Complete `step()` function in `EpsilonGreedy` class and report it in the written submission

In [ ]:
class EpsilonGreedy(MAB):
    def __init__(self, initial_actions: np.ndarray, initial_rewards: np.ndarray, epsilon: float):
        # ! DO NOT MODIFY THE FOLLOWING VARIABLES. You will use them in your implementation.
        super().__init__(initial_actions, initial_rewards)  # Initialize the parent class
        self.epsilon = epsilon  # Exploration rate

    def step(self) -> int:
        # TODO: COMPLETE THIS FUNCTION
        """
        Select an action via epsilon-greedy algorithm.

        Returns:
            int: Index of the selected action.
        """
        # YOUR CODE HERE
        raise NotImplementedError()


# Let's try an exploration rate of 0.1 and compare it with random model
epsilon = 0.1
e_greedy = EpsilonGreedy(initial_actions, initial_rewards, epsilon=epsilon)
e_greedy.update(true_rewards, n_steps)
e_greedy.plot_avg_reward(label=f"Epsilon={epsilon}")
full_random.plot_avg_reward(label='Random')

### ✰ 3.2(d) Complete `step()` function in `UCB` class and report it in the written submission

In [ ]:
class UCB(MAB):
    def __init__(self, initial_actions: np.ndarray, initial_rewards: np.ndarray, c: float):
        # ! DO NOT MODIFY THE FOLLOWING VARIABLES. You will use them in your implementation.
        super().__init__(initial_actions, initial_rewards)  # Initialize the parent class
        self.c = c

    def step(self) -> int:
        # TODO: COMPLETE THIS FUNCTION
        """
        Select an action via Upper Confidence Bound (UCB) algorithm.

        Returns:
            int: Index of the selected action.
        """
        # YOUR CODE HERE
        raise NotImplementedError()

### ✰ 3.2(e) Compare Different Algorithms

Now, impelemtent `EpsilonGreedy` and `UCB` with different hyperparameters and compare them. 

Produce a single plot that:
- have 3 `EpsilonGreedy` lines, 3 `UCB` lines, and 1 `FullRandom` line
- clearly indicate hyperparameters (up to your choice) you have experimented with in the legend

You can refer to the **toy vs random** plot we provide at the end of 3.2(b).

Include your plot in the written submission for Question 3.2(e)


In [ ]:
# YOUR CODE HERE
raise NotImplementedError()  # You can comment this line after having the required plot